In [1]:
import strahler
import os
os.environ["STRAHLER_PM_BACKEND"] = "simd"
from importlib.metadata import version as package_version
if package_version("strahler") != strahler.version():
    raise RuntimeError("Package/library version mismatch. Reinstall strahler and restart the kernel.")
print("strahler package and library versions match")

from IPython.display import HTML, display
from notebook_support import HERE, table
from lexicographic_nester import BACKENDS, host_svg
from family_benchmarks import (ORDERS, recorded, result_text, fixed_maps, shape_html)


strahler package and library versions match


# CABP_SWP: grow the protocol data size

CABP_SWP compares two descriptions of a communication protocol. We use data
sizes **2 and 4**, keeping capacity and window size at 1, and compare branching
and strong bisimulation separately.

The branching data-size-2 game is the CABP_SWP benchmark selected for
[SYNTCOMP 2026](https://github.com/SYNTCOMP/benchmarks/tree/4105caf1f1e5fd3b76657879bfce8021d130cbde/parity/keiren).
The other three cases extend that comparison using Keiren's protocol family.
All four use the same Knor conversion. Their priorities normalize to {0,1,2},
so increasing the data size changes the graph while the host heights stay fixed.


In [2]:
info, recording = recorded("cabp")
games = info["games"]
table(["case", "parameter", "vertices", "edges", "AD h E/O", "STRPM h E/O", "AD k cap", "STRPM k cap", "t cap"],
      [[g["label"], g["parameter"], g["vertices"], g["edges"], g["AD_height"],
        [d + 1 for d in g["PM_depth"]], g["AD_k_max"], g["PM_k_max"], g["t_max"]]
       for g in games])

case,parameter,vertices,edges,AD h E/O,STRPM h E/O,AD k cap,STRPM k cap,t cap
"branching-bisim, data=2",2,875281,1469041,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",19
"strong-bisim, data=2",2,125084,202028,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",16
"branching-bisim, data=4",4,2512705,4230209,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",21
"strong-bisim, data=4",4,339444,551252,"[2, 2]","[2, 3]","[2, 2]","[2, 3]",18


## Candidate hosts

In $U^k_{t,h}$, k bounds Strahler number, t sets the leaf allowance, and h
counts levels including the root. Here are the hosts at **(k,t)=(2,3)**.
AD uses height 2 for both players; STRPM uses heights 2 and 3.


In [3]:
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">' +
             host_svg(2, 3, 2, "AD E/O and STRPM Even: h=2") +
             host_svg(2, 3, 3, "STRPM Odd: h=3") + '</div>'))

## A winning decomposition

Zielonka's answer for the first game. The labels count the vertices in H, T
and S; the drawing keeps just the tree shape.


In [4]:
first_reference = next(r for r in recording["references"] if r["game"] == games[0]["id"])
display(HTML(shape_html(first_reference["reference_shape"])))


## Three search orders: one minute per case

We compare AD, symmetric AD and Strahler PM, each with and without iterative
shaving. Preprocessing and same-parity compression are off. PM uses the full
SIMD kernel, with scalar fallback for unary hosts.

Lexicographic order exhausts t before increasing k. Ratio starts at (1,1)
and favours balanced pairs. Table order tries smaller hosts first, using the
[precomputed size order](../HOST-ORDER.md), then falls back to ratio.

Runs have **60 seconds and 2 GiB**, with at most two solvers together.
Times include input, solving, output and winner checks against Zielonka.
Each cell shows the final pair and elapsed time. With +iter, the result also
includes wins from earlier rounds. After a method hits its limit, larger
instances in that series are left beyond its frontier.


In [5]:
by_case = {(r["game"], r["backend"], r["order"]): r for r in recording["rows"]}
for order in ORDERS:
    print(order)
    table(["case", "vertices", *BACKENDS],
          [[g["label"], g["vertices"],
            *[result_text(by_case[g["id"], b, order]) for b in BACKENDS]] for g in games])

lexicographic


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 0), 41.96 s",timeout (60 s cap),"(1, 0), 39.88 s",timeout (60 s cap),"(2, 3), 48.13 s"
"strong-bisim, data=2",125084,"(1, 0), 0.95 s","(1, 0), 0.96 s","(1, 0), 0.72 s","(1, 0), 0.76 s","(2, 5), 3.28 s","(2, 2), 2.83 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 0), 2.55 s","(1, 0), 2.59 s","(1, 0), 1.83 s","(1, 0), 1.89 s","(2, 5), 9.79 s","(2, 2), 9.37 s"


ratio


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 1), 41.58 s","(2, 3), 30.23 s","(1, 1), 32.56 s","(3, 7), 47.64 s","(2, 3), 26.23 s"
"strong-bisim, data=2",125084,"(1, 1), 1.21 s","(1, 1), 0.97 s","(1, 1), 0.72 s","(1, 1), 0.71 s","(3, 5), 1.88 s","(2, 2), 0.93 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),timeout (60 s cap),timeout (60 s cap),timeout (60 s cap),timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 1), 2.57 s","(1, 1), 2.62 s","(1, 1), 1.79 s","(1, 1), 1.86 s","(3, 5), 5.84 s","(2, 2), 2.34 s"


table


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
"branching-bisim, data=2",875281,timeout (60 s cap),"(1, 0), 41.75 s",timeout (60 s cap),"(1, 0), 37.72 s",timeout (60 s cap),"(2, 3), 35.71 s"
"strong-bisim, data=2",125084,"(1, 0), 0.97 s","(1, 0), 0.97 s","(1, 0), 0.72 s","(1, 0), 0.73 s","(2, 5), 3.13 s","(3, 2), 1.81 s"
"branching-bisim, data=4",2512705,beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap),beyond measured frontier,timeout (60 s cap)
"strong-bisim, data=4",339444,"(1, 0), 2.57 s","(1, 0), 2.76 s","(1, 0), 1.85 s","(1, 0), 1.89 s","(2, 5), 9.01 s","(3, 2), 5.39 s"


## Which fixed pairs solve the data-size-2 games?

We test AD-sym, AD-sym+iter, PM and PM+iter at fixed pairs: k=1…2 for AD,
k=1…3 for PM, and t=0…3. Each run has 60 seconds and 2 GiB.
With +iter, the same pair is repeated while shaving removes vertices.

Green cells solved; red cells finished partial; amber cells hit a limit.
Stars mark confirmed minimal pairs and diamonds mark candidates with smaller
pairs still unresolved. Gray cells were skipped because they cannot improve
on a measured solve.

Success is **not monotone in (k,t)**. We cap the scan by skipping larger
pairs after a solve. At fixed height, a solve at (k,t) transfers to (K,T) if
**K=k and T≥t**, or **K>k and T≥t+k−1**. A solved chain (k=1 or t=0)
transfers to every host. See the [embedding proof](../docs/host-monotonicity.md).

Green **⇒** cells follow from this bound and have no measured time. Other green symbols are measured solves. Gray cells remain unknown; amber cells
retain their measured limits.


In [6]:
maps, minima = fixed_maps(info, recording)
display(HTML(maps))
table(["case", "method", "minimal (k,t)", "time/memory limits"],
      [[case, method,
        ", ".join(str(pair) + ("" if pair in confirmed else " ?") for pair in observed) or "none",
        limits or "none"]
       for case, method, confirmed, observed, limits in minima])


case,method,"minimal (k,t)",time/memory limits
"branching-bisim, data=2",AD-sym,"(2, 3)",none
"branching-bisim, data=2",AD-sym+iter,"(1, 0)",none
"branching-bisim, data=2",PM,none,none
"branching-bisim, data=2",PM+iter,"(2, 3)",none
"strong-bisim, data=2",AD-sym,"(1, 0)",none
"strong-bisim, data=2",AD-sym+iter,"(1, 0)",none
"strong-bisim, data=2",PM,none,none
"strong-bisim, data=2",PM+iter,"(2, 2)",none


## Plain PM at larger t

Plain PM finishes partial throughout both t≤3 grids. Fresh fixed runs at
**(3,7)** for branching and **(3,5)** for strong bisimulation both solve.
These give working pairs; the smallest t remains open.


In [7]:
table(["case", "fixed (k,t)", "outcome", "seconds"],
      [[r["game"], (r["k"], r["t"]), r["outcome"], f"{r['seconds']:.2f}"]
       for r in recording["followup_rows"]])


case,"fixed (k,t)",outcome,seconds
cabp-branching-bisim-data2,"(3, 7)",solved,13.05
cabp-strong-bisim-data2,"(3, 5)",solved,0.77


Ratio solves the branching data-size-2 game with **five of six methods**;
lexicographic and table order each manage three. No measured method solves
the larger branching game within a minute. All methods and orders solve both
strong-bisimulation sizes.

Shaving makes a large difference to the fixed pairs. AD-sym+iter solves both
anchors at **(1,0)**. PM+iter reaches **(2,3)** for branching and **(2,2)**
for strong bisimulation. Plain PM needs larger t than this grid covers.

## Sources

The games come from [Keiren's CC0 dataset](https://doi.org/10.6084/m9.figshare.6004130.v1)
and the SYNTCOMP corpus. [cabp/README.md](cabp/README.md) describes their origins
and conversion; it also gives the commands for rerunning these measurements.
